# MISR Image Generation and CMPA Password Configuration on MCXC353

The MCXC353 boot ROM supports three application authentication schemes for an XiP (Execute
in Place) Master Boot Image (MBI): `plain`, `crc` and `misr`. The `misr` scheme relies on a
**Multiple Input Signature Register (MISR-128)** checksum that is calculated over the whole
application image and appended at the end of the image. The MISR calculation starts from a
128-bit **seed** value that must match the `IMG_MISR_SEED` field stored in the device's CMPA
(Customer Manufacturing Programmable Area) password region - if the seed used to build the
image does not match the seed programmed into CMPA, the ROM bootloader rejects the image.

This notebook shows how to:

1. Generate a MISR-authenticated MBI image from a plain application binary using *nxpimage*.
2. Generate the matching CMPA password (`CMPA_PSWD`) configuration and binary containing the
   `IMG_MISR_SEED` value using the *pfr* tool.
3. Verify that both artifacts are consistent (same seed) and inspect their binary layout.
4. Program both artifacts onto the device using SEGGER J-Link Commander, since MCXC353 does
   not support ISP mode.

## MISR image geometry

| Item | Value |
|------|-------|
| MISR algorithm | MISR-128, polynomial `X^128 + X^126 + X^101 + X^99 + 1` |
| Seed byte order | big-endian (as stored in CMPA) |
| Data alignment | 16 bytes (0x10) |
| Signature size | 16 bytes, appended at the end of the image |

## CMPA_PSWD layout (relevant fields)

| Offset | Field | Width |
|--------|-------|-------|
| 0x00 | PASSWORD_HEADER (marker `0x5057`) | 32b |
| 0x10 | DBG_AUTH_PASSWORD | 128b |
| 0x20 | **IMG_MISR_SEED** | 128b |
| 0x30 | ERASE_PASSWORD | 128b |
| 0x40 | PASSWORD_AREA_CRC32 | 32b |

## 1. Prerequisites

- SPSDK is needed with the examples extension: `pip install spsdk[examples]` (Please refer to
  the [installation](../../_knowledge_base/installation_guide.rst) documentation.)
- MCXC353 board
- A plain application binary compiled for XiP execution on MCXC353
  (`inputs/mcxc353_user_img_flash.bin` in this example).
- J-Link debugger connected (MCXC353 does not support ISP mode, so provisioning is done over
  the debug/SWD interface).

In [1]:
import glob
import os
import re
import shutil

from spsdk.utils.jupyter_utils import YamlDiffWidget

# This env variable sets colored logger output to STDOUT
%env JUPYTER_SPSDK=1
# Set a magic for command execution and echo
%alias execute echo %l && %l
%alias_magic ! execute

WORKSPACE = "workspace/"  # change this to path to your workspace
INPUTS = "inputs/"
VERBOSITY = (
    ""  # verbosity of commands, might be -v or -vv for debug or blank for no additional info
)

os.makedirs(WORKSPACE, exist_ok=True)

# Device configuration
FAMILY = "mcxc353"

# MISR seed shared between the MBI image and the CMPA_PSWD configuration.
# Both artifacts MUST use the same 128-bit seed, otherwise the ROM bootloader
# will reject the image during boot.
MISR_SEED = "12345678123456781234567812345678"

# Path to the SEGGER J-Link Commander executable used for programming the device.
# It is looked up on the system PATH first; if not found, the newest installation in the
# default SEGGER install location is used. Set JLINK_EXE manually to override, e.g.:
# JLINK_EXE = r"C:\Program Files\SEGGER\JLink_V844\JLink.exe"
JLINK_NAME = "JLink.exe" if os.name == "nt" else "JLinkExe"
JLINK_EXE = shutil.which(JLINK_NAME)
if not JLINK_EXE:
    SEGGER_DIRS = (
        [
            os.path.join(os.environ.get(var, ""), "SEGGER")
            for var in ("ProgramFiles", "ProgramFiles(x86)")
        ]
        if os.name == "nt"
        else ["/opt/SEGGER", "/Applications/SEGGER"]
    )
    candidates = [c for d in SEGGER_DIRS for c in glob.glob(os.path.join(d, "JLink*", JLINK_NAME))]
    # pick the newest version, e.g. JLink_V844 over JLink_V798h
    candidates.sort(
        key=lambda c: [int(n) for n in re.findall(r"\d+", os.path.basename(os.path.dirname(c)))]
    )
    JLINK_EXE = candidates[-1] if candidates else JLINK_NAME
print(f"J-Link Commander: {JLINK_EXE}")

env: JUPYTER_SPSDK=1
Created `%!` as an alias for `%execute`.
J-Link Commander: C:\Program Files\SEGGER\JLink_V844\JLink.exe


## 2. Master Boot Image (MISR) generation

Generation of the MBI is done with the *nxpimage* tool. A configuration file is required to
describe the input image, the output file and the MISR seed.

### 2.1 MBI configuration file

We have already prepared a MISR configuration file
([`inputs/mcxc353_xip_misr.yaml`](./inputs/mcxc353_xip_misr.yaml)) with the application path and
MISR seed filled in. Below, the diff widget regenerates the raw `nxpimage mbi get-templates`
template and highlights only the fields that were actually changed compared to the default
template - click **Show Diff** to see the comparison, or **Show User Config** to see the final
configuration file used in this example.

In [2]:
# Get difference between the raw template and our MISR configuration
YamlDiffWidget("inputs/mcxc353_misr_mbi.diffc").html

nxpimage mbi get-templates -f mcxc353 -o workspace/ --force 
Creating workspace/mcxc353_xip_plain.yaml template file.
Creating workspace/mcxc353_xip_crc.yaml template file.
Creating workspace/mcxc353_xip_misr.yaml template file.


### 2.2 MBI generation

Let's generate the MISR-authenticated Master Boot Image. The tool appends the 16-byte MISR-128
checksum (calculated over the whole aligned image, starting from `MISR_SEED`) at the end of the
plain application image and updates the image length/type flags in the Interrupt Vector Table
(IVT).

In [3]:
MBI_CONFIG_PATH = INPUTS + "mcxc353_xip_misr.yaml"
MBI_BIN = WORKSPACE + "mbi_misr.bin"

%! nxpimage $VERBOSITY mbi export -c $MBI_CONFIG_PATH

assert os.path.exists(MBI_BIN)

nxpimage  mbi export -c inputs/mcxc353_xip_misr.yaml 
Success. (Master Boot Image: workspace/mbi_misr.bin created.)


## 3. CMPA Password (CMPA_PSWD) configuration and binary

The `IMG_MISR_SEED` value used above must be programmed into the device's CMPA password region
so that the ROM bootloader starts the MISR calculation from the same seed when verifying the
image at boot time. Let's prepare the `CMPA_PSWD` configuration using the *pfr* tool.

### 3.1 CMPA_PSWD configuration file

We have already prepared a `CMPA_PSWD` configuration file
([`inputs/cmpa_pswd.yaml`](./inputs/cmpa_pswd.yaml)) with `IMG_MISR_SEED` set to the same value
used for the MBI generation above (the `DBG_AUTH_PASSWORD` and `ERASE_PASSWORD` fields are left
at their default, all-zero value in this example). As before, the diff widget regenerates the
raw `pfr get-template` template and highlights only the `IMG_MISR_SEED` field that was
changed.

In [4]:
# Get difference between the raw template and our CMPA_PSWD configuration
YamlDiffWidget("inputs/mcxc353_cmpa_pswd.diffc").html

pfr get-template -f mcxc353 -t cmpa_pswd -o workspace/cmpa_pswd_template.yaml --force 
The PFR cmpa_pswd template for mcxc353, Revision: latest has been saved into workspace/cmpa_pswd_template.yaml YAML file


### 3.2 Generate the CMPA_PSWD binary

The `pfr export` command generates the 68-byte `CMPA_PSWD` binary (64 bytes of register data +
4-byte CRC32 over the whole password region), ready to be written to the device's CMPA
password page at address `0x0100_2800`.

In [5]:
CMPA_PSWD_CONFIG = INPUTS + "cmpa_pswd.yaml"
CMPA_PSWD_BIN = WORKSPACE + "cmpa_pswd.bin"

%! pfr $VERBOSITY export -c $CMPA_PSWD_CONFIG -o $CMPA_PSWD_BIN

assert os.path.exists(CMPA_PSWD_BIN)

pfr  export -c inputs/cmpa_pswd.yaml -o workspace/cmpa_pswd.bin 
Success. (CMPA_PSWD binary has been generated and stored into workspace/cmpa_pswd.bin)


### 3.3 Inspect the CMPA_PSWD binary

Let's display the binary layout of the generated `CMPA_PSWD` page and confirm the
`IMG_MISR_SEED` field was written at offset `0x20`.

In [6]:
%! pfr $VERBOSITY info -f $FAMILY -t cmpa_pswd -b $CMPA_PSWD_BIN

pfr  info -f mcxc353 -t cmpa_pswd -b workspace/cmpa_pswd.bin 
BINARY IMAGE INFORMATION

Name:                    mcxc353 CMPA_PSWD
Description:             Customer Manufacturing Programmable Area - Password
Size:                    68 B (68 bytes, 0x44)
Offset:                  0x0 (0 bytes)
Absolute Address:        0x0
End Address:             0x43
Alignment:               1 bytes
Pattern:                 0x0
Sub-images Count:        8

VISUAL REPRESENTATION

+==0x0000_0000= mcxc353 CMPA_PSWD ======+
|              Size: 68 B               |
|  Customer Manufacturing Programmable  |
|            Area - Password            |
|             Pattern: 0x0              |
|+==0x0000_0000= PASSWORD_HEADER ======+|
||              Size: 4 B              ||
||           Password header           ||
|+==0x0000_0003========================+|
|+==0x0000_0004= Reserved 0x00004 =====+|
||              Size: 4 B              ||
|| This field is reserved for internal ||
||                 use        

## 4. Verify configuration consistency

As a final sanity check, let's parse the generated `CMPA_PSWD` binary back into YAML and
confirm the `IMG_MISR_SEED` value matches the seed used to build the MISR image in step 2.

In [7]:
from spsdk.utils.config import Config

# Seed used when building the MBI image (from the YAML configuration)
mbi_seed = Config.create_from_file(MBI_CONFIG_PATH)["misrSeed"]

# Seed programmed into the CMPA_PSWD binary (from the YAML configuration)
cmpa_seed = Config.create_from_file(CMPA_PSWD_CONFIG)["settings/IMG_MISR_SEED"]

print(f"MBI misrSeed         : {mbi_seed}")
print(f"CMPA_PSWD IMG_MISR_SEED: {cmpa_seed}")

assert mbi_seed.lower() == cmpa_seed.lower(), "MISR seed mismatch between MBI image and CMPA_PSWD!"
print("OK: MISR seed is consistent between the MBI image and the CMPA_PSWD configuration.")

MBI misrSeed         : 12345678123456781234567812345678
CMPA_PSWD IMG_MISR_SEED: 12345678123456781234567812345678
OK: MISR seed is consistent between the MBI image and the CMPA_PSWD configuration.


## 5. Programming the device

MCXC353 does **not** support ISP mode, so provisioning cannot be done with `blhost`/`pfr write`
over a UART/USB ISP connection. Instead, both artifacts must be written directly over the debug
(SWD) interface using SEGGER J-Link Commander's `LoadFile` command:

```
Erase 0x01002000, 0x01003FFF
LoadFile workspace/cmpa_pswd.bin 0x01002800
LoadFile workspace/mbi_misr.bin 0x0
```

The whole CMPA sector (`0x0100_2000`-`0x0100_3FFF`, 8 KB) is erased before the
`CMPA_PSWD` binary is written. Flash programming can only clear bits (`1` -> `0`), so
writing a new `IMG_MISR_SEED` over a previously programmed one without erasing would
store `old_seed AND new_seed` instead of the new value - the seed would no longer match
the MBI and the ROM bootloader would reject the image. Note that the erase also clears
`CMPA_CFG` (`0x0100_2000`) and `CMPA_LC` (`0x0100_3000`), which share the same sector.

The commands below have been captured into a J-Link Commander script
([`inputs/program_misr.jlink`](./inputs/program_misr.jlink)) that connects to the target, halts
the core, erases the CMPA sector, writes both binaries at their respective addresses, then
resets and starts the application:

In [8]:
with open(INPUTS + "program_misr.jlink", encoding="utf-8") as f:
    print(f.read())

// Copyright 2026 NXP
//
// SPDX-License-Identifier: BSD-3-Clause
//
// J-Link Commander script to program the MISR-authenticated Master Boot Image
// and the matching CMPA_PSWD (seed) binary onto the MCXC353 target.
//
// MCXC353 does not support ISP mode, so provisioning must be done over the
// debug interface (SWD) using J-Link Commander's LoadFile command instead of
// blhost/pfr write over an ISP connection.
//
// Run with (from the mcxc353_misr_image example directory):
//   JLink.exe -NoGui 1 -ExitOnError 1 -CommandFile inputs/program_misr.jlink
device MCXC353
si SWD          // Select interface
speed 4000      // Select speed
connect
r               // Reset
h               // Halt

// Erase the whole CMPA sector (0x01002000-0x01003FFF, 8 KB) before reprogramming.
// CMPA_CFG (0x01002000), CMPA_PSWD (0x01002800) and CMPA_LC (0x01003000) all
// live in this sector. Flash programming can only clear bits (1 -> 0), so without
// this erase a changed MISR seed would end up as AND(o

Connect the J-Link probe to the board, power it on, and run the script.

The `JLINK_EXE` variable set in the first cell is resolved automatically: `JLink.exe` is looked
up on the system `PATH` first, then in the default SEGGER install location
(`C:\Program Files\SEGGER\JLink_<version>\JLink.exe` on Windows), picking the newest version.
The resolved path is printed by the first cell. If J-Link is installed elsewhere, set
`JLINK_EXE` to the full path of the executable manually.

In [9]:
%! "$JLINK_EXE" -NoGui 1 -ExitOnError 1 -CommandFile "inputs/program_misr.jlink"

"C:\Program Files\SEGGER\JLink_V844\JLink.exe" -NoGui 1 -ExitOnError 1 -CommandFile "inputs/program_misr.jlink" 
SEGGER J-Link Commander V8.44 (Compiled Jun 18 2025 12:17:39)
DLL version V8.44, compiled Jun 18 2025 12:16:46

J-Link Commander will now exit on Error

J-Link Command File read successfully.
Processing script file...
J-Link>device MCXC353
J-Link connection not established yet but required for command.
Connecting to J-Link via USB...O.K.
Firmware: J-Link V11 compiled Apr  1 2025 10:02:30
Hardware version: V11.00
J-Link uptime (since boot): 0d 00h 14m 12s
S/N: 851006367
License(s): RDI, FlashBP, FlashDL, JFlash, GDB
USB speed mode: High speed (480 MBit/s)
VTref=3.315V
J-Link>si SWD
Selecting SWD as current target interface.
J-Link>speed 4000
Selecting 4000 kHz as target interface speed
J-Link>connect
Device "MCXC353" selected.


Connecting to target via SWD
InitTarget() start
******************************************************
J-Link script: MCX Cortex-M23 core J-Link scri

Once the script finishes, the device resets and starts running the flashed MISR-authenticated
application - the ROM bootloader recalculates the MISR checksum over the flashed image using
the `IMG_MISR_SEED` read from CMPA and compares it against the signature appended to the image.

## Summary

This notebook demonstrated the complete MISR provisioning flow for MCXC353:

1. Generated a MISR-authenticated Master Boot Image from a plain application binary using
   `nxpimage mbi export`.
2. Generated the matching `CMPA_PSWD` configuration and binary containing `IMG_MISR_SEED`
   using `pfr export`.
3. Verified that the MISR seed used in both artifacts is consistent, ensuring the ROM
   bootloader will be able to validate the image at boot time.
4. Programmed both artifacts onto the device over SWD using J-Link Commander, since MCXC353
   does not support ISP mode.